### This is a trial to run the meta-learning agent in the trial environment

In [1]:
import os, sys
os.environ["PYTHONPATH"] = "/home/rifnyrachman7"
sys.path.append("/home/rifnyrachman7/_merlion/")
sys.path.append("/home/rifnyrachman7/_merlion/ds-project-messiah/src")
sys.path.append("/home/rifnyrachman7/")

In [2]:
from gymnasium.wrappers import TimeLimit
import gymnasium as gym
from pathlib import Path

# Import both modules
from _metamorl.algo_ray.rllib.algorithms.maml.merlion_main_multitasks import MERLION, MERLIONConfig
from _metamorl.algo_ray.rllib.algorithms.maml.callback_save_population import _find_archive , MERLIONWithSaveCallbacks

 #v2.3.1 #use algo from local repo
import ray
from ray import air, tune
from ray.tune.registry import register_env
from gymnasium.wrappers import TimeLimit
from ray.air import Checkpoint
from ray.rllib.algorithms.algorithm import Algorithm

# Import the required modules

from messiah.environment import Environment
from messiah.agents.processing import MaxAvailableProcessingAgent

import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

import tempfile
import pickle


#from utils.plot_utils import PlotUtils

In [3]:
#import generated environment
from _merlion.random_env import simple_env_merlion
simple_sc = simple_env_merlion.SimpleSC()
#simple_sc.save_filename = "/home/rifnyrachman7/_merlion/random_env/simple_reward.pkl"

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
def save_full_checkpoints_for_population(algo, out_dir: str):
    archive = _find_archive(algo)
    if archive is None:
        print("[SavePopulation] MERLION archive not found on algorithm; skipping.")
        return

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    pol = algo.get_policy()
    P = int(getattr(archive, "population_size", len(getattr(archive, "archive", []))))

    for i in range(P):
        item = archive.get_archive_item(i)
        if item is None:
            continue

        theta_i = item["theta"]
        w_i = np.asarray(item["weight"], dtype=np.float32)   # <- correct key

        # save the full RLlib checkpoint for policy i
        pol.set_weights(theta_i)
        ckpt_dir = out / f"checkpoint_policy_{i}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        path = algo.save(str(ckpt_dir))
        print(f"[SavePopulation] Saved RLlib checkpoint for i={i} → {path}")

        # save the associated weight next to it
        with open(ckpt_dir / "weight.json", "w") as f:
            json.dump([float(x) for x in w_i.ravel().tolist()], f)

In [6]:
ray.shutdown()
ray.init()

register_env("simple_sc", lambda cfg: simple_env_merlion.make_env(cfg))

rollout_fragment_length = 32 #default 32

2026-01-26 10:02:21,706	WARNING utils.py:640 -- Detecting docker specified CPUs. In previous versions of Ray, CPU detection in containers was incorrect. Please ensure that Ray has enough CPUs allocated. As a temporary workaround to revert to the prior behavior, set `RAY_USE_MULTIPROCESSING_CPU_COUNT=1` as an env var before starting Ray. Set the env var: `RAY_DISABLE_DOCKER_CPU_WARNING=1` to mute this warning.
2026-01-26 10:02:21,708	WARNING utils.py:652 -- Ray currently does not support initializing Raywith fractional cpus. Your num_cpus will be truncated from 13.68 to 13.
2026-01-26 10:02:21,709	WARNING services.py:1780 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 67096576 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=10.24gb' to 'docker run' (or add it to the run_options list in a Ray clu

### Adaptation and Metalearning Phase

In [7]:
seed_base = 99

config = (
    MERLIONConfig()
    .rollouts(
        num_rollout_workers=4, rollout_fragment_length=rollout_fragment_length
    )
    .framework("torch")
    .environment("simple_sc", clip_actions=True)
    .training(
        inner_adaptation_steps=4,
        meta_batch_size=4, # follow rollout_workers in MIRACL that represent task batch number
        maml_optimizer_steps=10,
        gamma=0.99,
        lambda_=1.0,
        lr=0.001,
        vf_loss_coeff=0.5,
        inner_lr=0.03,
        use_meta_env=True,
        clip_param=0.3,
        kl_target=0.01,
        kl_coeff=0.001,
        model=dict(fcnet_hiddens=[64, 64]),
        train_batch_size=rollout_fragment_length,
        
        #MERLION specific parameters
        population_size = 10, #default # TO DO: ablation for this
        num_objectives = 3,
    )
    .evaluation(
        evaluation_num_workers=1,
        evaluation_interval=100,
        enable_async_evaluation=True,
    )
    .callbacks(MERLIONWithSaveCallbacks)
)

In [8]:
local_dir = "/home/rifnyrachman7/_merlion/training_data/simple_sc"
num_iterations = 48 #default 500

results = {}
runs = 10 #run multiple times for model robustness
for k in range(9, runs):
    print(f"RUNNING NUMBER-{k}")
    
    tuner = tune.Tuner(
        MERLION,
        param_space=config.to_dict(),
        run_config=air.RunConfig(
            name=f"merlion_simple_{k}",
            local_dir = local_dir,
            stop={"training_iteration": num_iterations},
            failure_config=air.FailureConfig(fail_fast="raise"),
            log_to_file=True,
            checkpoint_config=air.CheckpointConfig(checkpoint_at_end=True, num_to_keep=1),  # ensure a checkpoint exists
        )
    )
    results[k] = tuner.fit()
    
    # #save the best checkpoint for each run
    best = results[k].get_best_result()
    algo = Algorithm.from_checkpoint(best.checkpoint)
    
    out_dir = os.path.join(local_dir, f"metapolicies-{k}")
    os.makedirs(out_dir, exist_ok=True)
    save_full_checkpoints_for_population(algo, out_dir)

RUNNING NUMBER-7


/home/rifnyrachman7/.local/lib/python3.10/site-packages/ray/tune/execution/trial_runner.py:402: UserWarning: fail_fast='raise' detected. Be careful when using this mode as resources (such as Ray processes, file descriptors, and temporary files) may not be cleaned up properly. To use a safer mode, use fail_fast=True.
  warnings.warn(
(MERLION pid=1846) 2025-11-12 09:56:48,472	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
(MERLION pid=1846) 2025-11-12 09:56:48,472	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
(MERLION pid=1846) 2025-11-12 09:56:49,032	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.
(MERLION pid=1846) 2025-11-12 09:56:49,032	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' 

(MERLION pid=1846) 
(MERLION pid=1846) === POPULATION OBJECTIVE BREAKDOWN (Iter 0) ===
(MERLION pid=1846) 00 w=[0.382,0.302,0.316] obj=[   0.57,   0.50,   0.56] sc=    0.543  <- OK
(MERLION pid=1846) 01 w=[0.637,0.203,0.160] obj=[   0.63,   0.52,   0.52] sc=    0.590  <- OK
(MERLION pid=1846) 02 w=[0.113,0.409,0.479] obj=[   0.50,   0.53,   0.50] sc=    0.512  <- OK
(MERLION pid=1846) 03 w=[0.379,0.294,0.327] obj=[   0.56,   0.49,   0.50] sc=    0.519  <- OK
(MERLION pid=1846) 04 w=[0.484,0.135,0.381] obj=[   0.59,   0.53,   0.54] sc=    0.564  <- OK
(MERLION pid=1846) 05 w=[0.511,0.374,0.115] obj=[   0.54,   0.56,   0.52] sc=    0.546  <- OK
(MERLION pid=1846) 06 w=[0.283,0.083,0.634] obj=[   0.64,   0.55,   0.50] sc=    0.545  <- OK
(MERLION pid=1846) 07 w=[0.103,0.515,0.382] obj=[   0.61,   0.50,   0.49] sc=    0.508  <- OK
(MERLION pid=1846) 08 w=[0.154,0.748,0.099] obj=[   0.71,   0.47,   0.51] sc=    0.509  <- OK
(MERLION pid=1846) 09 w=[0.960,0.034,0.006] obj=[   0.46,   0.53,  

Trial name,adaptation_delta,agent_timesteps_total,connector_metrics,custom_metrics,date,done,episode_len_mean,episode_media,episode_reward_max,episode_reward_max_adapt_1,episode_reward_max_adapt_2,episode_reward_max_adapt_3,episode_reward_max_adapt_4,episode_reward_mean,episode_reward_mean_adapt_1,episode_reward_mean_adapt_2,episode_reward_mean_adapt_3,episode_reward_mean_adapt_4,episode_reward_min,episode_reward_min_adapt_1,episode_reward_min_adapt_2,episode_reward_min_adapt_3,episode_reward_min_adapt_4,episodes_this_iter,episodes_total,experiment_id,hostname,info,iterations_since_restore,node_ip,num_faulty_episodes,num_healthy_workers,perf,pid,policy_reward_max,policy_reward_mean,policy_reward_min,sampler_perf,time_since_restore,time_this_iter_s,time_total_s,timers,timestamp,timesteps_since_restore,timesteps_this_iter,timesteps_total,training_iteration,trial_id,warmup_time
MERLION_simple_sc_e56d5_00000,0.45499,0,"{'ObsPreprocessorConnector_ms': 0.005025523049490792, 'StateBufferConnector_ms': 0.0034943081083751862, 'ViewRequirementAgentConnector_ms': 0.08357422692435128}",{},2025-11-12_10-49-38,True,100,{},65.4859,61.1165,62.9445,63.8198,64.6545,62.7715,55.2726,59.187,61.5737,63.2265,58.5549,44.0371,55.901,59.6595,62.1662,210,10080,feda73578e054b3f8bb6a8b1ea0b922b,workspace-580f8f1b-0605-48d2-b4f5-8608dc1d24e9-0,"{'learner': {'default_policy': {'learner_stats': {'allreduce_latency': 0.0, 'grad_gnorm': 1.0, 'worker_loss': 30.212356567382812}, 'model': {}, 'custom_metrics': {}, 'num_agent_steps_trained': 2000, 'num_grad_updates_lifetime': 1152, 'diff_num_grad_updates_vs_sampler_policy': 9.0}}, 'num_steps_sampled': 1081000, 'num_steps_trained_this_iter': 23000, 'num_steps_trained': 1081000}",48,10.131.62.82,0,5,"{'cpu_util_percent': 8.089247311827956, 'ram_util_percent': 12.5}",1846,{},{},{},"{'mean_raw_obs_processing_ms': 0.3910643660689777, 'mean_inference_ms': 0.7241867419139063, 'mean_action_processing_ms': 0.1349443013978398, 'mean_env_wait_ms': 1.5416495635389316, 'mean_env_render_ms': 0.0}",3158.46,65.7474,3158.46,{},1762944578,1104000,23000,1104000,48,e56d5_00000,10.7731


(MERLION pid=1846) [MERLION] timesteps_this_iter=23000, timesteps_total=23000
(MERLION pid=1846) 
(MERLION pid=1846) === POPULATION OBJECTIVE BREAKDOWN (Iter 1) ===
(MERLION pid=1846) 00 w=[0.960,0.034,0.006] obj=[   0.48,   0.48,   0.52] sc=    0.478  <- OK
(MERLION pid=1846) 01 w=[0.181,0.666,0.153] obj=[   0.54,   0.50,   0.47] sc=    0.500  <- OK
(MERLION pid=1846) 02 w=[0.548,0.266,0.187] obj=[   0.57,   0.47,   0.45] sc=    0.520  <- OK
(MERLION pid=1846) 03 w=[0.395,0.534,0.071] obj=[   0.48,   0.49,   0.47] sc=    0.483  <- OK
(MERLION pid=1846) 04 w=[0.141,0.702,0.158] obj=[   0.48,   0.51,   0.46] sc=    0.500  <- OK
(MERLION pid=1846) 05 w=[0.817,0.045,0.138] obj=[   0.59,   0.47,   0.46] sc=    0.563  <- OK
(MERLION pid=1846) 06 w=[0.197,0.763,0.041] obj=[   0.64,   0.48,   0.54] sc=    0.516  <- OK
(MERLION pid=1846) 07 w=[0.181,0.446,0.373] obj=[   0.56,   0.49,   0.51] sc=    0.512  <- OK
(MERLION pid=1846) 08 w=[0.364,0.511,0.125] obj=[   0.55,   0.51,   0.49] sc=    0.

...

(RolloutWorker pid=11238) 2025-11-12 14:37:11,803	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=11238) 2025-11-12 14:37:11,803	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=11238) 2025-11-12 14:37:11,803	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=11238) 2025-11-12 14:37:11,803	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(R

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.


/home/rifnyrachman7/_merlion/fine_tuning/evaluate_merlion.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


     [1/100] R=21.17  vec=(23.53,48.68,11.41)
     [2/100] R=26.21  vec=(30.03,48.98,16.88)
     [3/100] R=24.03  vec=(28.01,48.32,14.21)
     [4/100] R=30.69  vec=(34.75,47.38,22.75)
     [5/100] R=32.78  vec=(36.17,47.03,25.79)
     [6/100] R=37.25  vec=(42.81,47.32,30.25)
     [7/100] R=37.78  vec=(43.17,48.73,30.34)
     [8/100] R=35.65  vec=(41.42,47.11,28.08)
     [9/100] R=39.64  vec=(47.68,47.98,31.50)
     [10/100] R=38.26  vec=(43.65,48.49,31.23)
     [11/100] R=36.92  vec=(41.23,48.41,30.22)
     [12/100] R=37.76  vec=(43.84,48.63,30.10)
     [13/100] R=38.30  vec=(44.77,48.89,30.61)
     [14/100] R=40.13  vec=(47.80,48.70,32.11)
     [15/100] R=41.08  vec=(47.71,47.98,34.04)
     [16/100] R=41.60  vec=(48.54,50.36,33.96)
     [17/100] R=39.41  vec=(46.70,50.69,30.99)
     [18/100] R=39.61  vec=(46.35,51.62,31.24)
     [19/100] R=36.92  vec=(41.82,51.99,28.62)


No such comm target registered: jupyter.widget.control
No such comm: 4e62d54f-f5e2-41a6-8dd4-b127a1996641


### METAMORL
timesteps = W (S + 1) H

### MERLION
H = max(timestep/episode, rollout_fragment_length)
S = inner_adaptation_steps
W = num_rollout_workers
P = population_size
J = meta_batch_size

timesteps = W (S + 1) H + P (J (S + 1) H + H)
            {metaMORL-like} {population-stage}

A timestep represents an agent-environment interaction sampling.

In MERLION, the task batch size is defined in meta_batch_size, while num_rollout_workers only determine the parallel computation for speed. In Metamorl, the n_tasks is defined based on num_rollout_worker, so it is attached to the core number used.

In MERLION, the iteration is calculated for the overall population, hence each individual solution learns much less iteration than Metamorl. The justification is because each individual solution learns locally, focusing within narrower search space.

### How the iteration works:

each MERLION individual θᵢ gets fewer total updates than the single MetaMORL meta-policy,

“each individual solution learns locally, focusing within a narrower search space”

because the sample budget is spread across many individuals rather than concentrated on one.

So:

As a theoretical / structural statement: say

“Per global iteration, MERLION trains all individuals; cost scales with P×meta_batch_size.”

As a fair-sample comparison statement: say

“For a fixed environment step budget, each MERLION individual effectively sees fewer updates than a single MetaMORL meta-policy, trading depth per policy for coverage/diversity across the population.”

PS: Total running time in MERLION might more than MetaMORL since aside from metapolicy update (inner and outer), it also involves weight update thru evolutionary mechanism.

NOTE: Currently, MERLION still contains MetaMORL-like update which is actually algorithmically unnecessary but needed. Hence, these stage can be ignored for iteration and timing.

What happens if you “remove Phase 1”? If by “remove Phase 1” you mean: delete inner_adaptation_steps transform, stop producing (samples, metrics_dict), and leave MERLIONMetaUpdate.__call__ untouched,

then:

data_tuple will no longer be a 2-tuple, _shared_samples = data_tuple[0] will be wrong, you’ll get shape/TypeErrors → so yes, bugs.

This is not because of Ray API, but because your execution_plan and MERLIONMetaUpdate are coupled.

#### For now, let's just do interpolation to ignore MetaMORL update and upgrade version later maybe.

### Try to fine-tune using stable-baselines3

In [8]:
from stable_baselines3 import PPO
from stable_baselines3.ppo.policies import MlpPolicy
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from _merlion.fine_tuning.ft_simple_env_merlion import TestSimpleSC
# from _metamorl.algo_ray.rllib.algorithms.maml import merlion_finetuning_sb3 as ft
from _metamorl.algo_ray.rllib.algorithms.maml import merlion_finetuning_sb3_independent as ft

In [9]:
# (Re)start Ray cleanly
ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False)

def make_env_ray(config):
    """
    RLlib factory: receives a dict `config` and must return a *Gymnasium Env instance*.
    We wrap with TimeLimit here so both RLlib and Gym share the same horizon behavior.
    """
    base = TestSimpleSC(**config)            # pass config keys into your env
    max_steps = int(config.get("max_steps", 100))
    return TimeLimit(base, max_episode_steps=max_steps)

register_env("testSC", make_env_ray)
register_env("simple_sc", lambda cfg: simple_env_merlion.make_env(cfg)) # register meta-env

2026-01-26 10:20:43,245	WARNING utils.py:640 -- Detecting docker specified CPUs. In previous versions of Ray, CPU detection in containers was incorrect. Please ensure that Ray has enough CPUs allocated. As a temporary workaround to revert to the prior behavior, set `RAY_USE_MULTIPROCESSING_CPU_COUNT=1` as an env var before starting Ray. Set the env var: `RAY_DISABLE_DOCKER_CPU_WARNING=1` to mute this warning.
2026-01-26 10:20:43,246	WARNING utils.py:652 -- Ray currently does not support initializing Raywith fractional cpus. Your num_cpus will be truncated from 13.68 to 13.
2026-01-26 10:20:43,248	WARNING services.py:1780 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 67096576 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=10.24gb' to 'docker run' (or add it to the run_options list in a Ray clu

In [10]:
# Optional: re-register safely if re-running cells
try:
    gym.spec("testSC")  # raises if not registered
    # Gymnasium 0.29.x: remove existing spec to allow re-register
    del gym.envs.registration.registry["testSC"]
except Exception:
    pass

def make_env_gym(**kwargs):
    base = TestSimpleSC(**kwargs)
    max_steps = int(kwargs.get("max_steps", 100))
    return TimeLimit(base, max_episode_steps=max_steps)

gym.register(
    id="testSC",
    entry_point=make_env_gym,   # callable that returns an Env
)

Finetuning timesteps = P x (M + 1) x K x W

P: population size

M: number of local perturbation

K: env steps per perturbation

W: rollout workers

In [16]:
# Train the remaining unfinished finetuning
k = 3
start_time = time.time()
ft.finetune_with_local_perturbations(
    pop_dir=f"/home/rifnyrachman7/_merlion/training_data/simple_sc/metapolicies-{k}",
    env_id="testSC",
    out_dir=f"/home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-{k}",
    total_steps=100, # default 5000
    record_every=1, # default 1
    eval_episodes=10, # default 50
    m_perturb=4,
    eps=0.05,
    seed=7,
    skip_existing=True,

)
end_time = time.time()
print(f"fine-tuning run-{k}:",end_time - start_time,"seconds")


=== Meta-policy 0 ===


(RolloutWorker pid=9110) 2025-11-19 16:16:48,597	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=9110) 2025-11-19 16:16:48,597	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=9110) 2025-11-19 16:16:48,597	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=9110) 2025-11-19 16:16:48,597	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(Rollo

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.
[skip+load] i000_m-1: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-3/i000_m-1 already exists, loading existing results into summaries.
[skip+load] i000_m00: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-3/i000_m00 already exists, loading existing results into summaries.
[skip+load] i000_m01: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-3/i000_m01 already exists, loading existing results into summaries.
[skip+load] i000_m02: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-3/i000_m02 already exists, loading existing results into summaries.
[skip+load] i000_m03: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-3/i000_m03 already exists, loading existing results into summaries.

=== Meta-policy 1 ===


(RolloutWorker pid=9513) 2025-11-19 16:16:57,300	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=9513) 2025-11-19 16:16:57,300	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=9513) 2025-11-19 16:16:57,300	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=9513) 2025-11-19 16:16:57,300	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(Rollo

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.
[skip+load] i001_m-1: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-3/i001_m-1 already exists, loading existing results into summaries.
[skip+load] i001_m00: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-3/i001_m00 already exists, loading existing results into summaries.
[skip+load] i001_m01: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-3/i001_m01 already exists, loading existing results into summaries.
[skip+load] i001_m02: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-3/i001_m02 already exists, loading existing results into summaries.
[skip+load] i001_m03: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-3/i001_m03 already exists, loading existing results into summaries.

=== Meta-policy 2 ===


...


=== Meta-policy 9 ===


(RolloutWorker pid=13593) 2025-11-19 18:08:40,115	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=13593) 2025-11-19 18:08:40,115	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=13593) 2025-11-19 18:08:40,115	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=13593) 2025-11-19 18:08:40,115	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(R

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.


/home/rifnyrachman7/_merlion/fine_tuning/evaluate_merlion.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Saved:
- /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-3/all_offspring_summary.csv
- /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-3/all_timesteps_long.csv
fine-tuning run-3: 10460.890636205673 seconds


In [8]:
#finetuning timesteps = 10 x 5 x 100 x 1 = 5000 timesteps
runs = 10
for k in range(runs):
    start_time = time.time()
    ft.finetune_with_local_perturbations(
        pop_dir=f"/home/rifnyrachman7/_merlion/training_data/simple_sc/metapolicies-{k}",
        env_id="testSC",
        out_dir=f"/home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-{k}",
        total_steps=100, # with sb3 n_steps = 512, default 5000
        record_every=1, # default 1
        eval_episodes=10, # default 50
        m_perturb=4,
        eps=0.05,
        seed=7,
        skip_existing=True  ,
    )
    end_time = time.time()
    print(f"fine-tuning run-{k}:",end_time - start_time,"seconds")

2025-11-20 10:20:16,949	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
2025-11-20 10:20:16,949	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
2025-11-20 10:20:16,964	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.
2025-11-20 10:20:16,964	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.



=== Meta-policy 0 ===


(RolloutWorker pid=28407) 2025-11-20 10:20:20,032	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=28407) 2025-11-20 10:20:20,032	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=28407) 2025-11-20 10:20:20,032	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=28407) 2025-11-20 10:20:20,032	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
20

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.
[skip+load] i000_m-1: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-0/i000_m-1 already exists, loading existing results into summaries.
[skip+load] i000_m00: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-0/i000_m00 already exists, loading existing results into summaries.
[skip+load] i000_m01: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-0/i000_m01 already exists, loading existing results into summaries.
[skip+load] i000_m02: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-0/i000_m02 already exists, loading existing results into summaries.
[skip+load] i000_m03: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-0/i000_m03 already exists, loading existing results into summaries.

=== Meta-policy 1 ===


(RolloutWorker pid=28799) 2025-11-20 10:20:29,381	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=28799) 2025-11-20 10:20:29,381	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=28799) 2025-11-20 10:20:29,381	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=28799) 2025-11-20 10:20:29,381	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(R

...

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.
[skip+load] i007_m-1: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i007_m-1 already exists, loading existing results into summaries.
[skip+load] i007_m00: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i007_m00 already exists, loading existing results into summaries.
[skip+load] i007_m01: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i007_m01 already exists, loading existing results into summaries.
[skip+load] i007_m02: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i007_m02 already exists, loading existing results into summaries.
[skip+load] i007_m03: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i007_m03 already exists, loading existing results into summaries.

=== Meta-policy 8 ===


(RolloutWorker pid=72105) 2025-11-20 23:41:59,156	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=72105) 2025-11-20 23:41:59,156	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=72105) 2025-11-20 23:41:59,157	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=72105) 2025-11-20 23:41:59,157	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(R

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.
[skip+load] i008_m-1: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i008_m-1 already exists, loading existing results into summaries.
[skip+load] i008_m00: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i008_m00 already exists, loading existing results into summaries.
[skip+load] i008_m01: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i008_m01 already exists, loading existing results into summaries.
[skip+load] i008_m02: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i008_m02 already exists, loading existing results into summaries.
[skip+load] i008_m03: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i008_m03 already exists, loading existing results into summaries.

=== Meta-policy 9 ===


(RolloutWorker pid=72445) 2025-11-20 23:42:07,820	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=72445) 2025-11-20 23:42:07,820	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=72445) 2025-11-20 23:42:07,821	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=72445) 2025-11-20 23:42:07,821	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(R

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.
[skip+load] i009_m-1: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i009_m-1 already exists, loading existing results into summaries.
[skip+load] i009_m00: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i009_m00 already exists, loading existing results into summaries.
[skip+load] i009_m01: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i009_m01 already exists, loading existing results into summaries.
[skip+load] i009_m02: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i009_m02 already exists, loading existing results into summaries.
[skip+load] i009_m03: /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i009_m03 already exists, loading existing results into summaries.
Saved:
- /home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/all_offspring_summary.csv
- /home/rifnyrachman7/_me

### Load the model to print policy

In [35]:
# Choose the highest HV : run-9 solution 38 (the most balanced weight: 0.38, 0.29, 0.33)

MODEL_PATH = "/home/rifnyrachman7/_merlion/training_data/simple_sc/finetuning_sb3-9/i007_m-1/sb3_model.zip"  # <- your file
ENV_ID     = "testSC"
run = 9

env = gym.make(ENV_ID, max_steps=100)
model = PPO.load(MODEL_PATH, env=env)

# Try to obtain human-readable action labels, otherwise use a[i]
action_labels = None
ue = getattr(env, "unwrapped", env)
for attr in ("get_action_names", "action_names", "action_labels"):
    if hasattr(ue, attr):
        names = getattr(ue, attr)
        action_labels = names() if callable(names) else names
        break

CSV_OUT    = f"/home/rifnyrachman7/_merlion/visualisation/action_sim_merlion-{run}.csv"
obs, info = env.reset()
rows = []
t = 0
while True:
    action, _ = model.predict(obs, deterministic=False)
    action = np.array(action).ravel()

    row = {"t": t}
    if action_labels and len(action_labels) == action.size:
        for name, val in zip(action_labels, action):
            row[name] = float(val)
    else:
        for i, val in enumerate(action):
            row[f"a{i}"] = float(val)

    # Optional: capture reward vector if the env provides it
    mo = info.get("mo_reward")
    if mo is not None and len(mo) >= 3:
        row["profit"]    = float(mo[0])
        row["emission"]  = float(mo[1])
        row["inequality"]= float(mo[2])

    rows.append(row)

    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
    t += 1

env.close()

df = pd.DataFrame(rows)
df.to_csv(CSV_OUT, index=False)
print(f"Saved action plan to {CSV_OUT}")

Saved action plan to /home/rifnyrachman7/_merlion/visualisation/action_sim_merlion-9.csv
